## Parking Sensor Coverage
Imports


In [1]:
# Data manipulation
import pandas as pd
import numpy as np

# Spatial analysis
import geopandas as gpd
from shapely.geometry import Point

# Visualisation
import matplotlib.pyplot as plt

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

##Load the street spatial dataset


In [2]:
# Load street spatial dataset

streets = gpd.read_file("streets_spatial.gpkg")
study_area = gpd.read_file("street_spatial_study_area.gpkg")


streets.head()

/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:200: RuntimeWarning: GPKG: unrecognized user_version=0x00000000 (0) on 'street_spatial_study_area.gpkg'
  return ogr_read(


,street_name,street_segment_id,suburb,postcode,lga,treatment_or_control,intervention_type,cbd,metro,regional,geometry
0,KEILOR ROAD,211.0,Essendon,3040,Moonee Valley (C),control,control,0.0,1.0,0.0,"LINESTRING (2491114.443 2417873.878, 2491077.3..."
1,KEILOR ROAD,212.0,Niddrie,3042,Moonee Valley (C),control,control,0.0,1.0,0.0,"LINESTRING (2490510.786 2418174.272, 2490453.5..."
2,VICTORIA AVENUE,224.0,Albert Park,3206,Port Phillip (C),control,control,0.0,1.0,0.0,"LINESTRING (2495452.168 2405934.307, 2495484.1..."
3,MALING ROAD,269.0,Canterbury,3126,Boroondara (C),control,control,0.0,1.0,0.0,"LINESTRING (2507070.632 2408393.017, 2507081.7..."
4,STEPHENSONS ROAD,300.0,Mount Waverley,3149,Monash (C),control,control,0.0,1.0,0.0,"LINESTRING (2511307.487 2402518.689, 2511325.0..."


## Statistics

In [3]:
print("Number of street segments:", len(streets))

print("\nColumns:")
print(streets.columns.tolist())

print("\nCoordinate Reference System:")
print(streets.crs)

Number of street segments: 315

Columns:
['street_name', 'street_segment_id', 'suburb', 'postcode', 'lga', 'treatment_or_control', 'intervention_type', 'cbd', 'metro', 'regional', 'geometry']

Coordinate Reference System:
EPSG:7899


##Treatment vs. control counts

In [4]:
streets["treatment_or_control"].value_counts()

,count
treatment_or_control,
control,219
treatment,96


In [5]:
streets[
    streets["treatment_or_control"]=="treatment"
]["intervention_type"].value_counts()

,count
intervention_type,
protected bike lane,80
regional_reallocation,10
pedestrian,6


##Load the parking sensor data

In [6]:
parking_file = "on-street-parking-bay-sensors.csv"

parking = pd.read_csv(parking_file)

parking.head()

,lastupdated,status_timestamp,zone_number,status_description,kerbsideid,location
0,2024-12-30T00:44:37+00:00,2024-08-18T08:23:46+00:00,7394.0,Unoccupied,9344,"-37.80494402936792, 144.95916129121264"
1,2025-01-10T02:44:36+00:00,2024-08-14T09:22:05+00:00,7392.0,Unoccupied,9373,"-37.80329074425858, 144.95836336946644"
2,2024-12-04T23:44:37+00:00,2024-11-28T02:30:57+00:00,7084.0,Present,8735,"-37.80223335664963, 144.96120480793184"
3,2024-12-04T23:44:37+00:00,2024-11-27T23:23:40+00:00,7084.0,Unoccupied,8749,"-37.80230361629087, 144.9618505999636"
4,2025-01-09T05:44:36+00:00,2023-08-21T00:09:03+00:00,7800.0,Present,24505,"-37.79756250415984, 144.95759881813447"


In [7]:
print(parking.columns.tolist())

['lastupdated', 'status_timestamp', 'zone_number', 'status_description', 'kerbsideid', 'location']


##Parse timestamps
Converts the timestamp text columns into proper datetime objects so they can be sorted, filtered by year, etc


In [8]:
parking["status_timestamp"] = pd.to_datetime(
    parking["status_timestamp"]
)

parking["lastupdated"] = pd.to_datetime(
    parking["lastupdated"]
)

In [9]:
parking.isnull().sum()

,0
lastupdated,0
status_timestamp,0
zone_number,240
status_description,0
kerbsideid,0
location,0


##Split location into latitude/longitude
This splits it on the comma into two new text columns and converts them to floats so they can be used as coordinates.

In [10]:
parking[
    ["latitude","longitude"]
] = parking["location"].str.split(
    ",",
    expand=True
)

parking["latitude"] = parking["latitude"].astype(float)

parking["longitude"] = parking["longitude"].astype(float)

parking.head()

,lastupdated,status_timestamp,zone_number,status_description,kerbsideid,location,latitude,longitude
0,2024-12-30 00:44:37+00:00,2024-08-18 08:23:46+00:00,7394.0,Unoccupied,9344,"-37.80494402936792, 144.95916129121264",-37.804944,144.959161
1,2025-01-10 02:44:36+00:00,2024-08-14 09:22:05+00:00,7392.0,Unoccupied,9373,"-37.80329074425858, 144.95836336946644",-37.803291,144.958363
2,2024-12-04 23:44:37+00:00,2024-11-28 02:30:57+00:00,7084.0,Present,8735,"-37.80223335664963, 144.96120480793184",-37.802233,144.961205
3,2024-12-04 23:44:37+00:00,2024-11-27 23:23:40+00:00,7084.0,Unoccupied,8749,"-37.80230361629087, 144.9618505999636",-37.802304,144.961851
4,2025-01-09 05:44:36+00:00,2023-08-21 00:09:03+00:00,7800.0,Present,24505,"-37.79756250415984, 144.95759881813447",-37.797563,144.957599


##Build a GeoDataFrame of sensor points
Converts the plain DataFrame into a GeoDataFrame by creating a Point geometry for each row from (longitude, latitude).

In [11]:
parking_gdf = gpd.GeoDataFrame(
    parking,
    geometry=gpd.points_from_xy(
        parking["longitude"],
        parking["latitude"]
    ),
    crs="EPSG:4326"
)


parking_gdf.head()

,lastupdated,status_timestamp,zone_number,status_description,kerbsideid,location,latitude,longitude,geometry
0,2024-12-30 00:44:37+00:00,2024-08-18 08:23:46+00:00,7394.0,Unoccupied,9344,"-37.80494402936792, 144.95916129121264",-37.804944,144.959161,POINT (144.95916 -37.80494)
1,2025-01-10 02:44:36+00:00,2024-08-14 09:22:05+00:00,7392.0,Unoccupied,9373,"-37.80329074425858, 144.95836336946644",-37.803291,144.958363,POINT (144.95836 -37.80329)
2,2024-12-04 23:44:37+00:00,2024-11-28 02:30:57+00:00,7084.0,Present,8735,"-37.80223335664963, 144.96120480793184",-37.802233,144.961205,POINT (144.9612 -37.80223)
3,2024-12-04 23:44:37+00:00,2024-11-27 23:23:40+00:00,7084.0,Unoccupied,8749,"-37.80230361629087, 144.9618505999636",-37.802304,144.961851,POINT (144.96185 -37.8023)
4,2025-01-09 05:44:36+00:00,2023-08-21 00:09:03+00:00,7800.0,Present,24505,"-37.79756250415984, 144.95759881813447",-37.797563,144.957599,POINT (144.9576 -37.79756)


##Align coordinate reference systems
Confirms the two datasets are in different CRSs: streets in EPSG:7899 (metres), parking in EPSG:4326 (degrees). You can't do accurate distance/spatial operations mixing degrees and metres, so they must be aligned.


In [12]:
print("Street CRS:")
print(streets.crs)

print("\nParking CRS:")
print(parking_gdf.crs)

Street CRS:
EPSG:7899

Parking CRS:
EPSG:4326


Reprojects the parking points into the same CRS as the streets (EPSG:7899), so both datasets now use metres and can be spatially compared.

In [13]:
parking_gdf = parking_gdf.to_crs(
    streets.crs
)

##Buffer the street geometries
Creates a copy of the streets layer and replaces each street line with a 20-metre buffer polygon around it. Since streets are lines and sensors are points that won't sit exactly on the line, this buffer creates a "search zone", any sensor within 20 m of a street segment counts as belonging to that street.

In [14]:
street_buffer = streets.copy()

street_buffer["geometry"] = (
    street_buffer.geometry.buffer(20)
)

street_buffer.head()

,street_name,street_segment_id,suburb,postcode,lga,treatment_or_control,intervention_type,cbd,metro,regional,geometry
0,KEILOR ROAD,211.0,Essendon,3040,Moonee Valley (C),control,control,0.0,1.0,0.0,"POLYGON ((2491068.102 2417875.452, 2491053.463..."
1,KEILOR ROAD,212.0,Niddrie,3042,Moonee Valley (C),control,control,0.0,1.0,0.0,"POLYGON ((2490444.607 2418184.919, 2490444.583..."
2,VICTORIA AVENUE,224.0,Albert Park,3206,Port Phillip (C),control,control,0.0,1.0,0.0,"POLYGON ((2495467.461 2405993.962, 2495489.864..."
3,MALING ROAD,269.0,Canterbury,3126,Boroondara (C),control,control,0.0,1.0,0.0,"POLYGON ((2507071.227 2408416.892, 2507071.868..."
4,STEPHENSONS ROAD,300.0,Mount Waverley,3149,Monash (C),control,control,0.0,1.0,0.0,"POLYGON ((2511305.311 2402623.995, 2511307.215..."


##Spatial join: match sensors to streets
Performs a spatial join: for each parking sensor point, it finds which street buffer polygon(s) it falls within, and attaches that street's attributes (street_segment_id, street_name, lga, treatment_or_control, intervention_type) to the sensor row.
how="inner" means only sensors that fall inside at least one street buffer are kept

In [15]:
parking_joined = gpd.sjoin(
    parking_gdf,
    street_buffer[
        [
            "street_segment_id",
            "street_name",
            "lga",
            "treatment_or_control",
            "intervention_type",
            "geometry"
        ]
    ],
    how="inner",
    predicate="within"
)


parking_joined.head()

,lastupdated,status_timestamp,zone_number,status_description,kerbsideid,location,latitude,longitude,geometry,index_right,street_segment_id,street_name,lga,treatment_or_control,intervention_type
29,2025-01-22 03:44:37+00:00,2024-05-03 03:56:57+00:00,NaN,Unoccupied,22776,"-37.82337001046017, 144.96661886495528",-37.823370,144.966619,POINT (2497061.179 2408628.196),243,5638.0,SOUTHBANK BOULEVARD,Melbourne (C),treatment,protected bike lane
33,2025-01-22 03:44:37+00:00,2024-05-03 04:21:09+00:00,NaN,Unoccupied,22772,"-37.823420177807634, 144.9668770118377",-37.823420,144.966877,POINT (2497083.907 2408622.636),243,5638.0,SOUTHBANK BOULEVARD,Melbourne (C),treatment,protected bike lane
35,2025-06-09 23:44:34+00:00,2025-06-07 23:44:32+00:00,7674.0,Present,51834,"-37.81433784981002, 144.968802858015",-37.814338,144.968803,POINT (2497253.125 2409630.718),116,5192.0,RUSSELL STREET,Melbourne (C),control,control
36,2025-06-09 23:44:34+00:00,2025-06-03 03:36:50+00:00,7674.0,Present,51824,"-37.8145733924308, 144.9689087192393",-37.814573,144.968909,POINT (2497262.455 2409604.579),116,5192.0,RUSSELL STREET,Melbourne (C),control,control
53,2025-05-08 05:44:34+00:00,2025-04-14 02:23:15+00:00,7218.0,Present,57938,"-37.82037346768652, 144.95463055373497",-37.820373,144.954631,POINT (2496005.589 2408960.338),118,5458.0,SPENCER STREET,Melbourne (C),control,control


In [16]:
print(
    "Total parking sensor observations:",
    len(parking_gdf)
)

print(
    "Sensors matched to study streets:",
    len(parking_joined)
)

Total parking sensor observations: 3391
Sensors matched to study streets: 1077


##Count distinct sensors per street
Groups the joined data by street segment and counts the number of unique sensors (kerbsideid) found near each one. This tells you how many distinct parking bays have sensor coverage per street.

In [17]:
street_sensor_coverage = (
    parking_joined
    .groupby("street_segment_id")
    ["kerbsideid"]
    .nunique()
    .reset_index()
)

street_sensor_coverage.columns = [
    "street_segment_id",
    "number_of_sensors"
]


street_sensor_coverage.head()

,street_segment_id,number_of_sensors
0,3828.0,2
1,3829.0,2
2,3830.0,5
3,3831.0,7
4,3833.0,10


##Merge sensor counts back onto all streets
Left-merges the sensor counts onto the full list of 315 streets (not just the ones with sensors), so streets with no matching sensors keep a row.
fillna(0) replaces missing counts (streets with no sensors nearby) with 0 rather than leaving them blank.

In [18]:
streets_coverage = streets.merge(
    street_sensor_coverage,
    on="street_segment_id",
    how="left"
)


streets_coverage[
    "number_of_sensors"
] = streets_coverage[
    "number_of_sensors"
].fillna(0)


streets_coverage.head()

,street_name,street_segment_id,suburb,postcode,lga,treatment_or_control,intervention_type,cbd,metro,regional,geometry,number_of_sensors
0,KEILOR ROAD,211.0,Essendon,3040,Moonee Valley (C),control,control,0.0,1.0,0.0,"LINESTRING (2491114.443 2417873.878, 2491077.3...",0.0
1,KEILOR ROAD,212.0,Niddrie,3042,Moonee Valley (C),control,control,0.0,1.0,0.0,"LINESTRING (2490510.786 2418174.272, 2490453.5...",0.0
2,VICTORIA AVENUE,224.0,Albert Park,3206,Port Phillip (C),control,control,0.0,1.0,0.0,"LINESTRING (2495452.168 2405934.307, 2495484.1...",0.0
3,MALING ROAD,269.0,Canterbury,3126,Boroondara (C),control,control,0.0,1.0,0.0,"LINESTRING (2507070.632 2408393.017, 2507081.7...",0.0
4,STEPHENSONS ROAD,300.0,Mount Waverley,3149,Monash (C),control,control,0.0,1.0,0.0,"LINESTRING (2511307.487 2402518.689, 2511325.0...",0.0


##Overall coverage summary

In [19]:
coverage_summary = pd.DataFrame({

    "Total Streets":
    [len(streets_coverage)],

    "Streets With Sensors":
    [
        (streets_coverage["number_of_sensors"] > 0)
        .sum()
    ],

    "Coverage Percentage":
    [
        round(
            (
                (streets_coverage["number_of_sensors"] > 0)
                .sum()
                /
                len(streets_coverage)
            )*100,
            2
        )
    ]

})


coverage_summary

,Total Streets,Streets With Sensors,Coverage Percentage
0,315,43,13.65


##Coverage by treatment vs. control
Groups streets by treatment_or_control and computes, for each group, the total street count, how many have sensors, and the resulting coverage percentage.

In [20]:
treatment_coverage = (
    streets_coverage
    .groupby("treatment_or_control")
    .agg(
        total_streets=("street_segment_id","count"),
        streets_with_sensors=
        ("number_of_sensors",
         lambda x:(x>0).sum())
    )
)


treatment_coverage[
    "coverage_percentage"
] = (
    treatment_coverage["streets_with_sensors"]
    /
    treatment_coverage["total_streets"]
    *100
)


treatment_coverage

,total_streets,streets_with_sensors,coverage_percentage
treatment_or_control,,,
control,219,24,10.958904
treatment,96,19,19.791667


##Coverage by intervention type
Same idea, but restricted to treatment streets only and broken down by intervention_type

In [21]:
intervention_coverage = (
    streets_coverage[
        streets_coverage["treatment_or_control"]
        =="treatment"
    ]
    .groupby("intervention_type")
    .agg(
        total_streets=
        ("street_segment_id","count"),

        streets_with_sensors=
        ("number_of_sensors",
        lambda x:(x>0).sum())
    )
)


intervention_coverage

,total_streets,streets_with_sensors
intervention_type,,
pedestrian,6,0
protected bike lane,80,19
regional_reallocation,10,0


##Coverage by local government area (LGA)
Breaks coverage down by council area (LGA). The first 10 rows shown are all 0 streets with sensors, meaning sensor coverage is concentrated in only a few LGAs (likely inner-city areas like Melbourne CBD, based on "LA TROBE STREET" appearing later), while most outer LGAs have none.

In [22]:
lga_coverage = (
    streets_coverage
    .groupby("lga")
    .agg(
        total_streets=
        ("street_segment_id","count"),

        streets_with_sensors=
        ("number_of_sensors",
        lambda x:(x>0).sum())
    )
)


lga_coverage.head(100)

,total_streets,streets_with_sensors
lga,,
Ballarat (C),7,0
Banyule (C),13,0
Bayside (C),6,0
Boroondara (C),20,0
Brimbank (C),2,0
Darebin (C),12,0
Glen Eira (C),19,0
Greater Bendigo (C),1,0
Greater Dandenong (C),4,0


##Add a year column to the joined data
Extracts the year from each sensor reading's timestamp for time-based analysis.Counts how many matched sensor observations fall in each year

In [23]:
parking_joined[
    "year"
] = parking_joined[
    "status_timestamp"
].dt.year


year_summary = (
    parking_joined
    .groupby("year")
    .size()
    .reset_index(name="observations")
)


year_summary

,year,observations
0,2022,1
1,2023,15
2,2024,86
3,2025,261
4,2026,714


##Parking utilisation rate per street
For each street segment, calculates the proportion of readings that were Present (occupied) vs Unoccupied, normalized to sum to 1 (i.e. a percentage split).
.unstack() pivots the result so Present and Unoccupied become separate columns instead of stacked rows

In [24]:
utilisation = (
    parking_joined
    .groupby(
        [
            "street_segment_id",
            "street_name"
        ]
    )
    ["status_description"]
    .value_counts(normalize=True)
    .unstack()
)


utilisation.head()

,status_description,Present,Unoccupied
street_segment_id,street_name,,
3828.0,LA TROBE STREET,0.500000,0.500000
3829.0,LA TROBE STREET,0.500000,0.500000
3830.0,LA TROBE STREET,0.600000,0.400000
3831.0,LA TROBE STREET,0.857143,0.142857
3833.0,LA TROBE STREET,0.600000,0.400000


##Export results

In [25]:
parking_joined.to_csv(
    "teamB_parking_sensor_street_mapping.csv",
    index=False
)


streets_coverage.to_file(
    "street_parking_sensor_coverage.gpkg",
    driver="GPKG"
)